## Tutorial

# Calculate and Compare Canopy Water Content

## Second of two notebooks

### Authors: Hannah Rieder, Randi Neff, Bridget Hass

In this tutorial, we will learn how to evaluate forest health using a calculation of the Canopy Water Content (CWC) from individual tiles at the Soaproot Saddle (SOAP) field site in the Sierra National Forest in California. The hyperspectral data for the CWC calculation comes from the National Ecological Observatory Network's (NEON) Level 3 Spectrometer orthorectified surface directional reflectance - mosaic data product and the Earth Surface Mineral Dust Source Investigation (EMIT) L2A Reflectance Data Product.

## The objectives of this tutorial (divided between two notebooks) are to:

* Use co-located data from NEON and EMIT
* Calculate Canopy Water Content (CWC) from NEON and EMIT hyperspectral data
* Evaluate CWC data at different scales
* Compare between burned and unburned areas

DATA The data provided with this tutorial were derived from existing code at:

* NEON Spectrometer orthorectified surface bidirectional reflectance data.
* Shapefiles for Creek fire boundary and NEON burned and unburned tiles which are found in the DATA folder.
* EMIT L2A Estimated Surface Reflectance granule(s) that cover the NEON burned and unburned tiles.
* Land Processes Distributed Active Archive Center (LP DAAC).

Additional data will be downloaded programmatically within this tutorial.

## What we should have after completing notebook 1:

* two EMIT cropped datasets (one for the burned tile and one for the unburned tile) exported to netcdf files
* two NEON reflectance datasets (one for the burned tile and one for the unburned tile). These datasets will already have been converted from hdf5 format into xarray, have the scale factor applied, have bad bands set to NaN, have necessary data types turned from float64 to float32, and be exported to netcdf files
* **make sure we have all 4 scenarios**

## Tutorial Outline for Notebook 2 - Canopy Water Content Comparison

1. Setup
2. Open NEON and EMIT Reflectance Data
3. Calculate Canopy Water Content (CWC)
4. Compare CWC Datasets

## Reference/credit to:
the [3 Equivalent Water Thickness/Canopy Water Content from Imaging Spectroscopy Data](https://nasa.github.io/VITALS/python/03_EMIT_CWC_from_Reflectance.html).

In [1]:
# Import Packages
import os, sys #python module to create and acces file paths
import pathlib
# Some cells may generate warnings that we can ignore.
# Comment below lines to see.
import warnings
warnings.filterwarnings('ignore')

import numpy as np #work with multi-dimensional arrays
import xarray as xr #work with labelled multi-dimenstional arrays
from osgeo import gdal #work with raster and vector geospatial data
import rasterio as rio #work with geospatial raster data
import rioxarray as rxr #work with raster arrays
from matplotlib import pyplot as plt #plotting data
import hvplot.xarray #plot multi-dimensional arrays
import hvplot.pandas #plot DataFrames/Series
import pandas as pd #work with DataFrames
import geopandas as gpd #work with geospatial shapefiles
import earthaccess #search for, download, & stream NASA earth data
from tqdm.notebook import tqdm # Progress bars on loops
import requests
from scipy.optimize import least_squares #nonlinear least-squares

#import neonutilities as nu #work with NEON reflectance data
import h5py #work with NEON reflectance data

### 1. Setup

In the setup section, we will do three things:
1. create directories to store the input and output data and scripts for this project,
2. download and import the extra necessary scripts needed for this tutorial, and
3. download a comma separated value (CSV) file needed for the CWC calculation function.

### 1.1 Create Data and Scripts (modules) Directories

The directories we will make are: an overarching data directory, a reflectance data directory, a CWC data directory, and a modules directory.

The overarching data directory will contain the reflectance and CWC data directories and a CSV file containing lab measurements of the complex refractive index of liquid water. 

The reflectance data directory should contain the two EMIT cropped datasets (one for the burned tile and one for the unburned tile) and the two NEON reflectance datasets (one for the burned tile and one for the unburned tile) created in Tutorial Notebook 01. All of these datasets should be NetCDF files.

The CWC data directory will be where we store the results of this tutorial notebook: the CWC calculations of the burned and unburned tiles calculated using the cropped EMIT and NEON reflectance data.

The modules directory must be in the same folder as where you have these tutorial notebooks stored for some of the imported functions to work. In this modules directory, we will manually download some Python scripts (.py files). The scripts contain various functions we'll use in this tutorial.

The CWC calculation functions (calc_ewt and calc_ewt_neon) are expecting the data and the k_liquid_water_ice.csv file to be stored in a directory that is at the same file level as where you have this notebook stored. In the cell below, the `data_dir = r"../data"` code ensures that the data directory will be at the same file level as where this tutorial notebook is stored.

Here is a visual of how the directory and file structure will look once these directories are created and files are downloaded:

```
project-root/
│
├── data/                     # Main folder for data
│   ├── cwc/                  # Subfolder for canopy water content (CWC) data genearted in tutorial_notebook_02
│   │   ├── emit_burn_cwc.nc               
│   │   ├── emit_burn_cwc.tif               
│   │   ├── emit_unburn_cwc.nc
│   │   ├── emit_unburn_cwc.tif                           
│   │   ├── neon_burn_cwc.nc               
│   │   ├── neon_burn_cwc.tif               
│   │   ├── neon_unburn_cwc.nc               
│   │   └── neon_unburn_cwc.tif          
│   │
│   ├── refl/                 # Subfolder for reflectance data generated in tutorial_notebook_01
│   │   ├── emit_burn_refl.nc
│   │   ├── emit_unburn_refl.nc
│   │   ├── neon_burn_refl_float32.nc
|   |   └── neon_unburn_refl_float32.nc
|   └── k_liquid_water_ice.csv
│
└── notebooks/                # Subfolder for modules and tutorial notebooks
    ├── modules/              # Subfolder for Python scripts for processing and analysis
    │   ├── .ipynb_checkpoints
    |   ├── __pycache__
    |   ├── __init__
    |   ├── ewt_tools.py
    |   ├── ewt_calc2.py
    |   └── test_functions.py
    ├── tutorial_notebook_01.ipynb
    └── tutorial_notebook_02.ipynb
```

In [7]:
# Define the file path for the data directory
data_dir = r"../data"

# Create the data_dir if it doesn't already exist
if not os.path.exists(data_dir):
    os.makedirs(data_dir)
    print(f'data directory made here: {data_dir}')
else:
    print(f'data directory already exists here: {data_dir}')

data directory already exists here: ../data


In [8]:
# List of directories names to make
dir_list = ["refl", "cwc"]

# Define the root path where the directories will be created
root_path = data_dir

# Use a for loop to create the directories in the dir_list
for dir_name in dir_list:
    full_path = os.path.join(root_path, dir_name)
    if not os.path.exists(full_path):
        os.makedirs(full_path)
        print(f'directory made here: {full_path}')
    else:
        print(f'directory already exists here: {full_path}')

directory already exists here: ../data\refl
directory already exists here: ../data\cwc


In [9]:
# Define the file path for the modules directory
modules_dir = r"./modules"

# Create the data_dir if it doesn't already exist
if not os.path.exists(modules_dir):
    os.makedirs(modules_dir)
    print(f'modules directory made here: {modules_dir}')
else:
    print(f'modules directory already exists here: {modules_dir}')

modules directory already exists here: ./modules


### 1.2 Download and Import Necessary Scripts

The Python scripts that we will put in the modules directory (ewt_calc2.py, emit_tools.py, and test_functions.py) contain functions that will allow us to calculate and visualize CWC and see a directory's contents. The emit_tools.py script is required for the functions in the ewt_calc2.py script to work. We can access and download the scripts with the following steps:

1. Follow [this link](https://github.com/NEONScience/AOP-EMIT/blob/hrieder/notebooks/exploratory/hr/modules/ewt_calc2.py) to access the ewt_calc2.py script. **MUST UPDATE THIS ONCE FINAL, CORRECT A/O 07/26/2025!!!**
2. Manually download the raw file to your computer.
3. Move the ewt_calc2.py script into the modules directory we made above (`modules_dir = r"./modules"`). **It is important that the ewt_calc2.py script is in the modules_dir; the import code below expects the script to be in the modules_dir.**
4. Run the code in the cell below to import functions from the ewt_calc2.py script into this notebook.
5. Repeat steps 1-4 except, for step 1, follow [this link](https://github.com/NEONScience/AOP-EMIT/blob/hrieder/notebooks/exploratory/hr/modules/test_functions.py) to access the test_functions.py script.**MUST UPDATE THIS ONCE FINAL, CORRECT A/O 07/26/2025!!!**
6. Repeat steps 1-4 except, for step 1, follow [this link](https://github.com/NEONScience/AOP-EMIT/blob/hrieder/notebooks/exploratory/hr/modules/emit_tools.py) to access the emit_tools.py script.**MUST UPDATE THIS ONCE FINAL, CORRECT A/O 07/26/2025!!!**

In [10]:
# Import functions from the python scripts in the modules directory
from modules.test_functions import data_download_tracker, surfrfl_hvplot_image
from modules.emit_tools import emit_xarray #open EMIT datasets into xarray.Dataset
from modules.ewt_calc2 import calc_ewt, calc_ewt_neon #canopy water content fxn

### 1.3 Download and Open the Refractive Index of Liquid Water per Wavelength CSV

The CWC calculation functions (calc_ewt and calc_ewt_neon) requires this CSV file to work. The name of the CSV file is k_liquid_water_ice.csv, which can be found in the EMIT VITALS GitHub repository data folder. We can access and download the k_liquid_water_ice.csv file with the following steps:

1. Follow [this link](https://github.com/nasa/VITALS/blob/main/data/k_liquid_water_ice.csv) to access the k_liquid_water_ice.csv file in the EMIT VITALS repository.
2. Manually download the raw file to your computer.
3. Move the k_liquid_water_ice.csv file into the data directory we made above (`data_dir = r"../data"`). **It is important that the k_liquid_water_ice.csv file is in the data_dir; the calc_ewt and calc_ewt_neon functions expect the CSV file to be in the data_dir.**
4. Run the code in the cell below to open and look at the k_liquid_water_ice.csv file in this notebook. 

The existing [EMIT VITALS CWC tutorial notebook](https://nasa.github.io/VITALS/python/03_EMIT_CWC_from_Reflectance.html#setup) has a helpful details about what this k_liquid_water_ice.csv file is:
>We need some lab measurements of the complex refractive index of liquid water to obtain the wavelength-dependent absorption coefficients. They are calculated by taking four times the product of Pi and the imaginary part of the refractive index, divided by wavelength. The refractive index of liquid water per wavelength is provided by the k_liquid_water_ice.csv in the data folder.

In [11]:
# Define file path to k_liquid_water_ice.csv file
wp_fp = ("../data/k_liquid_water_ice.csv")

# Read k_liquid_water_ice.csv file into a DataFrame
k_wi = pd.read_csv(wp_fp)

# Check k_wi DataFrame
k_wi.head()

,wvl_1,T = 22°C,wvl_2,T = -8°C,wvl_3,T = -25°C,wvl_4,T = -7°C,wvl_5,T = 25°C (H),wvl_6,T = 20°C,wvl_7,T = 25°C (S),Index
0,666.7,2.470000e-08,NaN,NaN,NaN,NaN,660.0,1.660000e-08,650.0,1.640000e-08,650.0,1.870000e-08,650.12971,1.674130e-08,0
1,667.6,2.480000e-08,NaN,NaN,NaN,NaN,670.0,1.890000e-08,675.0,2.230000e-08,651.0,1.890000e-08,654.63616,1.777420e-08,1
2,668.4,2.480000e-08,NaN,NaN,NaN,NaN,680.0,2.090000e-08,700.0,3.350000e-08,652.0,1.910000e-08,660.69347,1.939950e-08,2
3,669.3,2.520000e-08,NaN,NaN,NaN,NaN,690.0,2.400000e-08,725.0,9.150000e-08,653.0,1.940000e-08,665.27314,2.031380e-08,3
4,670.2,2.530000e-08,NaN,NaN,NaN,NaN,700.0,2.900000e-08,750.0,1.560000e-07,654.0,1.970000e-08,669.88461,2.097930e-08,4


### 2. Open NEON and EMIT Reflectance Data

In [12]:
# Define filepaths to the cropped NEON reflectance NetCDF files
neon_burn_fp = ("../data/REFL/neon_burn_refl_float32.nc")

#neon_unburn_fp = ("../data/REFL/neon_unburn_refl.nc")

In [13]:
# Define filepaths to the cropped EMIT reflectance NetCDF files
emit_burn_fp = ("../data/REFL"
                "/EMIT_L2A_RFL_001_20230731T205320_2321214_004_SOAP_burn.nc")
# emit_unburn_fp = ("../data/REFL/emit_unburn_refl.nc")

In [14]:
# Open NetCDF NEON & EMIT burned and unburned datasets
neon_burn_ds = xr.open_dataset(neon_burn_fp, decode_coords="all")
#neon_unburn_ds = xr.open_dataset(neon_unburn_fp, decode_coords="all")

emit_burn_ds = xr.open_dataset(emit_burn_fp, decode_coords="all")
# emit_unburn_ds = xr.open_dataset(emit_unburn_fp, decode_coords="all")

Check datasets

why - what are we checking for??

acknowledge that this isn't reproducible but for this purpose we're just opening them to make sure they look right before we complete the CWC calculation.

In [15]:
# Check neon_burn_ds
neon_burn_ds

<xarray.Dataset> Size: 2GB
Dimensions:           (x: 1000, y: 1000, wavelengths: 426)
Coordinates:
  * x                 (x) float64 8kB 2.98e+05 2.98e+05 ... 2.99e+05 2.99e+05
  * y                 (y) float64 8kB 4.1e+06 4.1e+06 ... 4.101e+06 4.101e+06
    fwhm              (wavelengths) float32 2kB ...
    good_wavelengths  (wavelengths) float32 2kB ...
    spatial_ref       int32 4B ...
  * wavelengths       (wavelengths) float32 2kB 383.9 388.9 ... 2.512e+03
Data variables:
    reflectance       (y, x, wavelengths) float32 2GB ...
Attributes:
    no_data_value:     -9999.0
    scale_factor:      10000.0
    bad_band_window1:  [1340 1445]
    bad_band_window2:  [1790 1955]
    projection:        +proj=UTM +zone=11 +ellps=WGS84 +datum=WGS84 +units=m ...
    spatial_ref:       PROJCS["WGS_1984_UTM_Zone_11N",GEOGCS["GCS_WGS_1984",D...
    EPSG:              32611

In [16]:
# Check neon_unburn_ds
#neon_unburn_ds

In [17]:
# Check emit_burn_ds
emit_burn_ds

<xarray.Dataset> Size: 457kB
Dimensions:           (wavelengths: 285, latitude: 18, longitude: 22)
Coordinates:
  * wavelengths       (wavelengths) float32 1kB 381.0 388.4 ... 2.493e+03
    fwhm              (wavelengths) float32 1kB ...
    good_wavelengths  (wavelengths) float32 1kB ...
  * latitude          (latitude) float64 144B 37.03 37.03 37.03 ... 37.03 37.02
  * longitude         (longitude) float64 176B -119.3 -119.3 ... -119.3 -119.3
    elev              (latitude, longitude) float32 2kB ...
    spatial_ref       int32 4B ...
Data variables:
    reflectance       (latitude, longitude, wavelengths) float32 451kB ...
Attributes: (12/40)
    ncei_template_version:             NCEI_NetCDF_Swath_Template_v2.0
    summary:                           The Earth Surface Mineral Dust Source ...
    keywords:                          Imaging Spectroscopy, minerals, EMIT, ...
    Conventions:                       CF-1.63
    sensor:                            EMIT (Earth Surface Mineral Dust Sourc...
    instrument:                        EMIT
    ...                                ...
    spatial_ref:                       GEOGCS["WGS 84",DATUM["WGS_1984",SPHER...
    geotransform:                      [-1.19978439e+02  5.42232520e-04 -0.00...
    day_night_flag:                    Day
    title:                             EMIT L2A Estimated Surface Reflectance...
    granule_id:                        EMIT_L2A_RFL_001_20230731T205320_23212...
    Orthorectified:                    True

In [18]:
# Check emit_unburn_cs
#emit_unburn_ds

### 3. Calculate Canopy Water Content (CWC)

#### Calculate CWC using the calc_ewt function imported in the beginning

In [12]:
# Learn about calc_ewt function
help(calc_ewt)

Help on function calc_ewt in module modules.ewt_calc2:

calc_ewt(filepath: str, outdir: str, n_cpu: int = 7, ewt_detection_limit: float = 0.5, return_cwc: bool = False, is_emit: bool = True) -> None
    This function will calculate the equivalent water thickness (EWT) or canopy water content (CWC) from an EMIT .nc reflectance
    file using `ray` for parallelization, orthorectify if necessary, and write a cloud-optimized geotiff output.



In [13]:
# Learn about neon_calc_ewt function
help(calc_ewt_neon)

Help on function calc_ewt_neon in module modules.ewt_calc2:

calc_ewt_neon(filepath: str, outdir: str, n_cpu: int = 7, ewt_detection_limit: float = 0.5, return_cwc: bool = False) -> None
    This function will calculate the equivalent water thickness (EWT) or canopy water content (CWC) from a NEON .nc reflectance
    file using `ray` for parallelization and write a cloud-optimized geotiff output. The NEON data are already be orthorectified
    and need to have a 'spatial_ref' as part of the ds.variables.keys().



In [14]:
# Set output directory where results of CWC function will be stored
out_dir = r"../data/cwc/"

#original out dir:
#emit_out_dir = "../../../data/SOAP/EMIT/CWC/"
#add neon output directory here too? maybe just go back to one output directory?
# make consistent w/ where the .nc files are coming from, also depends on whether file names for outputs are clear
# possible NEW out dir: emit_out_dir = "../data/output/EMIT/CWC/"

Below is non-conditional code to calculate CWC using the EMIT reflectance data cropped to the burned tile (emit_burn_fp) and unburned tile (emit_unburn_fp). If the %%time isn't the first thing in the cell, it doesn't work. Need to investigate the %%time code and see how necessary it is.

In [15]:
%%time
emit_burn_cwc_ds = calc_ewt(
    # Burned EMIT dataset file path
    emit_burn_fp,
    out_dir,
    ewt_detection_limit=1.5,
    return_cwc=True
)

# View emit_burn_cwc_ds 
emit_burn_cwc_ds

2025-07-26 18:01:10,075	INFO worker.py:1843 -- Started a local Ray instance. View the dashboard at 127.0.0.1:8265 


CPU times: total: 3.19 s
Wall time: 41.1 s


<xarray.Dataset> Size: 5kB
Dimensions:      (latitude: 18, longitude: 22)
Coordinates:
  * latitude     (latitude) float64 144B 37.03 37.03 37.03 ... 37.03 37.03 37.02
  * longitude    (longitude) float64 176B -119.3 -119.3 -119.3 ... -119.3 -119.3
    elev         (latitude, longitude) float32 2kB ...
    spatial_ref  int32 4B ...
Data variables:
    cwc          (latitude, longitude) float64 3kB nan nan nan ... 0.1897 0.1897
Attributes: (12/13)
    flight_line:            emit20230731t205320_o21214_s000
    time_coverage_start:    2023-07-31T20:53:20+0000
    time_coverage_end:      2023-07-31T20:53:32+0000
    easternmost_longitude:  -118.65918754030226
    northernmost_latitude:  37.399622098045
    westernmost_longitude:  -119.978439262086
    ...                     ...
    spatialResolution:      0.000542232520256367
    spatial_ref:            GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84...
    geotransform:           [-1.19978439e+02  5.42232520e-04 -0.00000000e+00 ...
    day_night_flag:         Day
    title:                  EMIT Estimated Equivalent Water Thickness (EWT) /...
    granule_id:             EMIT_L2A_RFL_001_20230731T205320_2321214_004

In [ ]:
# %%time
# emit_unburn_cwc_ds = calc_ewt(
#     # Unburned EMIT dataset file path
#     emit_unburn_fp,
#     out_dir,
#     ewt_detection_limit=1.5,
#     return_cwc=True
# )

# # View emit_unburn_cwc_ds 
# emit_unburn_cwc_ds

Below is non-conditional code to calculate CWC using the NEON reflectance data cropped to the burned tile (neon_burn_fp) and unburned tile (neon_unburn_fp). 

In [ ]:
%%time
neon_burn_cwc_ds = calc_ewt_neon(
    # Burned NEON dataset file path
    neon_burn_fp,
    out_dir,
    ewt_detection_limit=1.5,
    return_cwc=True
)

# View neon_burn_cwc_ds 
neon_burn_cwc_ds

In [ ]:
# %%time
# neon_unburn_cwc_ds = calc_ewt_neon(
#     # Unburned NEON dataset file path
#     neon_unburn_fp,
#     emit_out_dir,
#     ewt_detection_limit=1.5,
#     return_cwc=True
# )

# # View neon_unburn_cwc_ds 
# neon_unburn_cwc_ds

In [ ]:
# Start to a possible conditional statement for CWC calculation
cwc_soap_burn_fp = r'C:\Users\riede\Documents\EDA Capstone\AOP-EMIT\data\SOAP\EMIT\CWC\EMIT_L2A_RFL_001_20230731T205320_2321214_004_SOAP_burned_CWC.nc'
#if cwc has already been calculated and saved to a netcdf file,
if os.path.exists(cwc_soap_burn_fp):
    print('path exists')
    #open the CWC netcdf file and
    cwc_ds = xr.open_dataset(cwc_soap_burn_fp, decode_coords="all")
    #display the CWC dataset
    display(cwc_ds)
else:
    print('path does not exist, calculating CWC...')
    #commented the %%time out b/c it threw an error, CWC still calculated w/o it for the burned tile
    #%%time
    emit_burn_cwc_ds = calc_ewt(
        #burned emit dataset
        emit_burn_fp,
        out_dir,
        ewt_detection_limit=1.5,
        return_cwc=True
    )
    display(emit_burn_cwc_ds)

# The above code all works, it just needs to be changes so it is reproducible and not specific to my file paths!

In [ ]:
# %%time
# emit_burn_cwc_ds = calc_ewt(
#     #burned emit dataset
#     emit_burn_fp,
#     emit_out_dir,
#     ewt_detection_limit=1.5,
#     return_cwc=True
# )

# # View emit_burn_cwc_ds 
# emit_burn_cwc_ds

In [ ]:
# Export CWC datasets to NetCDF files to save the datasets
emit_burn_cwc_ds.to_netcdf("../data/cwc/emit_burn_cwc.nc")
# emit_unburn_cwc_ds.to_netcdf("../data/CWC/emit_unburn_cwc.nc")
neon_burn_cwc_ds.to_netcdf("../data/cwc/neon_burn_cwc.nc")
# neon_unburn_cwc_ds.to_netcdf("../data/CWC/neon_unburn_cwc.nc")

### 3.1 Visualize CWC Datasets

In [ ]:
# Plot CWC of the SOAP burned tile using surfrfl_hvplot_image fxn
surfrfl_hvplot_image(
    emit_burn_cwc_ds,
    plottitle=f"SOAP Burned Tile {emit_burn_cwc_ds.cwc.long_name} ({emit_burn_cwc_ds.cwc.units}) July 31, 2023",
clabel="Canopy Water Content (g/cm^2)")

### 4. Compare CWC Datasets

Start w/ histogram comparisons of values - see bridget's 07 notebook. Bridget also created som eKDE (Kernal density plots)

Then think about rescaling and then finding the difference

